# GPT-2 From Scratch

### Overview

This project implements a GPT-2 style decoder-only Transformer language model completely from scratch using PyTorch. The implementation is inspired by Andrej Karpathy's educational series while following a clean, modular software engineering approach.

### Objectives

- Build a GPT-2 style language model from scratch
- Understand every component of the Transformer architecture
- Train the model on a text dataset
- Generate coherent text
- Develop a Streamlit interface for inference

### Technologies Used

- Python
- PyTorch
- NumPy
- Matplotlib
- Streamlit
- Google Colab / VS Code

In [1]:
import os
import math
import random

import torch
import torch.nn as nn
from torch.nn import functional as F

import matplotlib.pyplot as plt

torch.manual_seed(42)

print("PyTorch Version:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch Version: 2.7.1+cpu
Device: CPU


In [2]:
# Hyperparameters

batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200

n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

# 1. Dataset Loading

In this section, we load the Tiny Shakespeare dataset, which will be used to train our GPT-2 model. This dataset consists of dialogues from Shakespeare's plays and serves as the training corpus for our language model.

In [3]:
DATA_PATH = "../data/input.txt"

In [4]:
# Load Dataset

with open(DATA_PATH, "r", encoding="utf-8") as file:
    text = file.read()

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [5]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



# 2. Dataset Statistics

Before building the tokenizer, let's understand the dataset by checking its size and the number of unique characters present.

In [6]:
print(f"Dataset Length : {len(text):,} characters")
print(f"Unique Characters : {len(set(text))}")

Dataset Length : 1,115,394 characters
Unique Characters : 65


In [7]:
chars = sorted(list(set(text)))
print(chars)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [8]:
print("Vocabulary Size:", len(chars))

Vocabulary Size: 65


# 3. Character Vocabulary

GPT does not understand raw text. The first step is to convert every unique character into a unique integer. This mapping forms the vocabulary used by the tokenizer.

In [9]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocabulary Size:", vocab_size)

Vocabulary Size: 65


In [10]:
# Character to Integer Mapping

stoi = {ch: i for i, ch in enumerate(chars)}

# Integer to Character Mapping

itos = {i: ch for i, ch in enumerate(chars)}

In [11]:
print(stoi)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [12]:
print(itos)

{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i', 48: 'j', 49: 'k', 50: 'l', 51: 'm', 52: 'n', 53: 'o', 54: 'p', 55: 'q', 56: 'r', 57: 's', 58: 't', 59: 'u', 60: 'v', 61: 'w', 62: 'x', 63: 'y', 64: 'z'}


# 4. Character Tokenizer

A tokenizer converts text into numerical representations that a neural network can process. In this implementation, we use a character-level tokenizer where every unique character is assigned a unique integer ID.

In [13]:
# Encode Function

encode = lambda s: [stoi[c] for c in s]

In [14]:
# Decode Function

decode = lambda l: ''.join([itos[i] for i in l])

In [15]:
sample_text = "Hello GPT"

encoded = encode(sample_text)

print("Original Text:")
print(sample_text)

print("\nEncoded:")
print(encoded)

Original Text:
Hello GPT

Encoded:
[20, 43, 50, 50, 53, 1, 19, 28, 32]


In [16]:
decoded = decode(encoded)

print("Decoded Text:")
print(decoded)

Decoded Text:
Hello GPT


In [17]:
print(sample_text == decoded)

True


In [18]:
sentence = "Machine Learning"

encoded_sentence = encode(sentence)
decoded_sentence = decode(encoded_sentence)

print(encoded_sentence)
print(decoded_sentence)

[25, 39, 41, 46, 47, 52, 43, 1, 24, 43, 39, 56, 52, 47, 52, 45]
Machine Learning
